# Categorical Compositionality Probe (v2 — asymmetric judge)

Tests whether an LLM satisfies the **functor law** `R(f ∘ g) = R(f) ∘ R(g)` on chained queries. Failure = a literal *compositional confabulation*: the model produced an answer to the chained query `f(g(x))` that doesn't match its answer to the composed query `(f ∘ g)(x)`, even though by the functor law they must agree.

Companion to the post [*Categorical reasoning is a benchmark, not a worldview*](https://coproduct.one). Runs in ~45 seconds on Colab; costs ~$0.20 in Anthropic API spend at default settings.

**Categorical setup.** For each triple `(entity, g, f)`:
- `g` is a query template like `"What is the capital of {x}?"` (a morphism `g: Entity → City`)
- `f` is a query template like `"What language is spoken in {y}?"` (a morphism `f: City → Language`)
- `f ∘ g` is the composed query `"What language is spoken in the capital of {x}?"`

We compute three answers from the model: `g(x)`, `f(g(x))` (chained), and `(f ∘ g)(x)` (composed). A **judge model** scores whether the chained and composed answers are the same fact. Violation rate = `1 − (#functorial / total)`.

**Why v2 matters: asymmetric judging.** v1 used the same model as both the system-under-test AND the judge — a known confound that biases the YES/NO toward self-agreement (the same model is more likely to think two of its own outputs mean the same thing). v2 defaults to `claude-sonnet-4-6` as the judge while testing `claude-haiku-4-5`, so the judge has independent factual coverage. The sanity-check cell re-runs the judge symmetrically (Haiku judging Haiku) and reports the disagreement rate so you can see exactly how much self-judgment was inflating v1's score.

In [ ]:
%pip install -q anthropic pandas

In [ ]:
import os, time, json
from dataclasses import dataclass
import anthropic
import pandas as pd

# API key: try Colab Secrets first, fall back to interactive prompt.
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    if not os.environ.get("ANTHROPIC_API_KEY"):
        os.environ["ANTHROPIC_API_KEY"] = input("Anthropic API key: ").strip()

client = anthropic.Anthropic()
print("client ready")

In [ ]:
# Models. The default keeps the judge ASYMMETRIC (stronger model) to dodge
# the self-agreement bias that inflated v1's functorial rate. The categorical
# claim from the post is that newer reasoning models hallucinate MORE on
# long chains — re-run with claude-opus-4-7 as the system under test to
# stress that prediction.
MODEL_UNDER_TEST       = "claude-haiku-4-5-20251001"
JUDGE_MODEL            = "claude-sonnet-4-6"          # asymmetric — stronger than the test model
JUDGE_MODEL_SYMMETRIC  = "claude-haiku-4-5-20251001"  # same as test — used by the sanity-check cell only

print(f"system under test: {MODEL_UNDER_TEST}")
print(f"asymmetric judge:  {JUDGE_MODEL}")
print(f"symmetric judge:   {JUDGE_MODEL_SYMMETRIC} (sanity-check only)")

In [ ]:
@dataclass
class Triple:
    name: str
    entity: str
    g: str        # template with {x} — first morphism
    f: str        # template with {y} — second morphism
    composed: str # template with {x} — the literal composition (f ∘ g)

# Each triple is hand-picked so the composition is unambiguous + the
# answer is a short string. Real benchmarks would pull (entity, g, f)
# from Wikidata via SPARQL — that's the obvious v3 move.
TRIPLES = [
    Triple(
        "capital→language",
        "France",
        "What is the capital of {x}? Answer with just the city name.",
        "What language is primarily spoken in {y}? Answer with just the language name.",
        "What language is primarily spoken in the capital of {x}? Answer with just the language name.",
    ),
    Triple(
        "birthcountry→continent",
        "Mozart",
        "In which country was {x} born? Answer with just the country name.",
        "On which continent is {y} located? Answer with just the continent name.",
        "On which continent was {x} born? Answer with just the continent name.",
    ),
    Triple(
        "city→currency",
        "Tokyo",
        "In which country is {x} located? Answer with just the country name.",
        "What is the currency of {y}? Answer with just the currency name.",
        "What is the currency of the country containing {x}? Answer with just the currency name.",
    ),
    Triple(
        "field→first_nobel",
        "Albert Einstein",
        "What was {x}'s primary scientific field? Answer with one word.",
        "Who was the first person awarded the Nobel Prize in {y}? Answer with just the full name.",
        "Who was the first person awarded the Nobel Prize in {x}'s primary scientific field? Answer with just the full name.",
    ),
    Triple(
        "company→ceo_alma_mater",
        "Apple Inc",
        "Who is the current CEO of {x}? Answer with just the person's full name.",
        "From which university did {y} earn their undergraduate degree? Answer with just the university name.",
        "From which university did the current CEO of {x} earn their undergraduate degree? Answer with just the university name.",
    ),
    Triple(
        "language→designer_employer",
        "Rust",
        "Who is the original designer of the {x} programming language? Answer with just the person's full name.",
        "Which company most recently employed {y} (as of late 2024)? Answer with just the company name.",
        "Which company most recently employed the original designer of the {x} programming language (as of late 2024)? Answer with just the company name.",
    ),
    Triple(
        "discoverer→nationality",
        "Pluto",
        "Who discovered {x} (the dwarf planet)? Answer with just the person's full name.",
        "What was {y}'s nationality? Answer with just the nationality.",
        "What was the nationality of the person who discovered {x} (the dwarf planet)? Answer with just the nationality.",
    ),
    Triple(
        "protocol→inventor_birth_year",
        "HTTP",
        "Who invented {x}? Answer with just the person's full name.",
        "In what year was {y} born? Answer with just the four-digit year.",
        "In what year was the inventor of {x} born? Answer with just the four-digit year.",
    ),
]

print(f"{len(TRIPLES)} triples loaded")

In [ ]:
def ask(prompt: str, model: str, max_tokens: int = 200) -> str:
    """One-shot single-turn query. Returns the trimmed text response."""
    r = client.messages.create(
        model=model,
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}],
    )
    return r.content[0].text.strip()

def judge_equiv(a: str, b: str, model: str) -> bool:
    """Asks the judge whether two short answers refer to the same fact.
    Returns True when the judge replies YES (case-insensitive prefix).
    Caller passes the model explicitly so we can re-run with a different
    judge in the symmetric-vs-asymmetric sanity-check cell."""
    prompt = (
        "Below are two short answers to the same factual question. "
        "Decide whether they refer to the same fact about the world. "
        "Tolerate trivial differences in wording (e.g. 'the USA' vs 'United States'). "
        "Reply with exactly one word: YES or NO.\n\n"
        f"Answer A: {a}\nAnswer B: {b}"
    )
    return ask(prompt, model=model, max_tokens=5).upper().startswith("YES")

In [ ]:
# Run the probe — 3 model calls per triple (g(x), f(g(x)) chained, (f∘g)(x) composed).
results = []
for i, t in enumerate(TRIPLES, 1):
    g_x         = ask(t.g.format(x=t.entity),        model=MODEL_UNDER_TEST)
    fg_chained  = ask(t.f.format(y=g_x),             model=MODEL_UNDER_TEST)
    fg_composed = ask(t.composed.format(x=t.entity), model=MODEL_UNDER_TEST)
    results.append({
        "name":        t.name,
        "entity":      t.entity,
        "g(x)":        g_x,
        "f(g(x))":     fg_chained,
        "(f∘g)(x)":    fg_composed,
    })
    print(f"[{i}/{len(TRIPLES)}] {t.name:<32}  chained={fg_chained!r:<32}  composed={fg_composed!r}")
    time.sleep(0.2)  # gentle on rate limits

print("\nrun complete")

In [ ]:
# Score every triple via the ASYMMETRIC judge (default: Sonnet 4.6 judging Haiku 4.5).
# This is the headline number.
for i, r in enumerate(results, 1):
    r["functorial_asym"] = judge_equiv(r["f(g(x))"], r["(f∘g)(x)"], model=JUDGE_MODEL)
    print(f"[{i}/{len(results)}] {r['name']:<32}  {'✓ functorial' if r['functorial_asym'] else '✗ VIOLATION'}")
    time.sleep(0.2)

## Sanity check: how much was the symmetric judge inflating v1?

Re-judge every triple with the same model that produced the answers. If the symmetric judge agrees with the asymmetric one on every triple, v1's confound was harmless on this sample. If they disagree on N triples, v1 was over-counting functoriality by `N / total` — which is exactly the bias term we wanted to expose.

*This cell adds 8 more API calls (~$0.01 in Haiku spend) and is safe to skip if you're rate-limited.*

In [ ]:
# Re-judge with the SYMMETRIC judge (same model as system under test).
# This is the v1 setting; we keep it ONLY to measure the self-agreement bias.
for i, r in enumerate(results, 1):
    r["functorial_sym"] = judge_equiv(r["f(g(x))"], r["(f∘g)(x)"], model=JUDGE_MODEL_SYMMETRIC)
    agree = "  AGREE" if r["functorial_sym"] == r["functorial_asym"] else "  DISAGREE"
    print(f"[{i}/{len(results)}] {r['name']:<32}  asym={'✓' if r['functorial_asym'] else '✗'}  sym={'✓' if r['functorial_sym'] else '✗'} {agree}")
    time.sleep(0.2)

disagreements = sum(1 for r in results if r["functorial_sym"] != r["functorial_asym"])
sym_inflation = sum(1 for r in results if r["functorial_sym"] and not r["functorial_asym"])
print(f"\n=== Judge-bias diagnostic ===")
print(f"Total disagreements: {disagreements} / {len(results)} ({disagreements/len(results):.1%})")
print(f"Symmetric-judge inflation: {sym_inflation} (cases where Haiku-judges-Haiku said ✓ but Sonnet said ✗)")
print(f"\nThe inflation count is the v1→v2 correction term. If it's 0, the asymmetric")
print(f"judge was overkill on this sample. If it's >0, v1 was over-counting functoriality")
print(f"(undercounting hallucination) by exactly that much.")

In [ ]:
df = pd.DataFrame(results)
violations_asym = (~df["functorial_asym"]).sum()
violations_sym  = (~df["functorial_sym"]).sum()
rate_asym = violations_asym / len(df)
rate_sym  = violations_sym  / len(df)
print(f"\n=== Functor-law violation rate ===")
print(f"Model under test: {MODEL_UNDER_TEST}")
print(f"")
print(f"Asymmetric judge ({JUDGE_MODEL}):  {violations_asym} / {len(df)}  =  {rate_asym:.1%}  ← headline")
print(f"Symmetric judge  ({JUDGE_MODEL_SYMMETRIC}):  {violations_sym} / {len(df)}  =  {rate_sym:.1%}  ← v1-style (biased toward self-agreement)")
print(f"")
print(f"A violation IS a categorical confabulation: f(g(x)) ≠ (f∘g)(x).")
print(f"Each FAIL row is a fact the model invented during composition.\n")
df[["name", "entity", "g(x)", "f(g(x))", "(f∘g)(x)", "functorial_asym", "functorial_sym"]]

## What to do with the result

1. **Sanity-check the FAIL rows in `functorial_asym`.** Each is a categorical confabulation: the model produced an answer to the chained query that doesn't match its answer to the composed query, even though the functor law says they must agree. Read each FAIL row carefully — some are real, some are judge errors, some are ambiguous prompts.
2. **Note the v1→v2 delta.** If `rate_sym < rate_asym`, the symmetric judge was inflating v1's functorial-rate (i.e. *understating* hallucination). The gap is the magnitude of the self-agreement confound, on this sample.
3. **Swap models.** Change `MODEL_UNDER_TEST` to `claude-opus-4-7` or `claude-sonnet-4-6` and re-run. The categorical prediction is that *reasoning-tier* models violate functoriality MORE than lighter ones because longer chains have more places to drop a type.
4. **Add triples.** This is 8 hand-picked entities. Real benchmarks would pull hundreds from Wikidata via SPARQL: any entity + two SPARQL-derivable predicates gives you a `(g, f)` pair for free.
5. **Sharpen the judge.** A single-call YES/NO judge is still noisy. The next move is a pairwise judge with chain-of-thought + a third "I DON'T KNOW" option (Lawvere-Tierney j!) — replace the binary verdict with the topos-theoretic Ω classifier we keep talking about.
6. **Compose with provenance.** A triple that violates functoriality AND can be traced to a specific contradicting source via the provenance fibration is a far more actionable signal than either alone. That's the next notebook.

## Cite this

Companion post: [*Categorical reasoning is a benchmark, not a worldview*](https://coproduct.one). Live demos of the runtime functor-law check at [topos-canvas.fly.dev/atom](https://topos-canvas.fly.dev/atom/).